In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
project_path = '/content/drive/MyDrive/CSCI 5527 Group Project/'
print("Files in project folder:")
print(os.listdir(project_path))

Files in project folder:
['Electronics.train.csv.gz', 'Electronics.valid.csv.gz', 'Electronics.test.csv.gz', 'Draft - idea.pdf', 'Yelp_Dataset.ipynb', 'Draft_notebook_dataset_exploration (1).ipynb', 'Lightning Talk.gslides', 'Progress Report Draft.gdoc', 'Lightning Talk Script.gdoc', 'Draft_notebook_dataset_exploration_(1).ipynb', 'Papers', 'fusion.ipynb', 'evaluation.ipynb', 'Final Report Draft.gdoc', 'Untitled (8).ipynb', 'embeddings.ipynb', 'beauty_rec', 'Beauty Amazon.ipynb']


In [ ]:
import os

beauty_path = os.path.join(project_path, 'beauty_rec')
print(os.listdir(beauty_path))

['amazon_beauty_2023_full.csv', 'amazon_beauty_cleaned.csv', 'amazon_beauty_2M.csv', 'text_embeddings.npy', 'image_embeddings.npy']


In [ ]:
import os

PROJECT_PATH = '/content/drive/MyDrive/CSCI 5527 Group Project/'
BEAUTY_PATH  = os.path.join(PROJECT_PATH, 'beauty_rec')

print('Project folder:', os.listdir(PROJECT_PATH))
print('Beauty folder :', os.listdir(BEAUTY_PATH))

Project folder: ['Electronics.train.csv.gz', 'Electronics.valid.csv.gz', 'Electronics.test.csv.gz', 'Draft - idea.pdf', 'Yelp_Dataset.ipynb', 'Draft_notebook_dataset_exploration (1).ipynb', 'Lightning Talk.gslides', 'Progress Report Draft.gdoc', 'Lightning Talk Script.gdoc', 'Draft_notebook_dataset_exploration_(1).ipynb', 'Papers', 'fusion.ipynb', 'evaluation.ipynb', 'Final Report Draft.gdoc', 'Untitled (8).ipynb', 'embeddings.ipynb', 'beauty_rec', 'Beauty Amazon.ipynb']
Beauty folder : ['amazon_beauty_2023_full.csv', 'amazon_beauty_cleaned.csv', 'amazon_beauty_2M.csv', 'text_embeddings.npy', 'image_embeddings.npy', 'interactions.csv', 'item_map.csv', 'user_map.csv']


In [ ]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv(
    os.path.join(BEAUTY_PATH, 'amazon_beauty_2M.csv'),
    low_memory=False
)
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

items = (
    df[['parent_asin', 'images_meta', 'description',
        'title_meta', 'features', 'categories']]
    .drop_duplicates('parent_asin')
    .reset_index(drop=True)
    .rename(columns={'images_meta': 'image_url'})
)
items['item_idx'] = np.arange(1, len(items) + 1)   # 1-based; 0 = pad

item_to_idx    = dict(zip(items['parent_asin'], items['item_idx']))
df['item_idx'] = df['parent_asin'].map(item_to_idx)
df['user_id']  = df['user_id'].astype(str)
uid, inv       = np.unique(df['user_id'], return_inverse=True)
df['user_idx'] = inv

df = df.sort_values(['user_idx', 'timestamp']).reset_index(drop=True)
df['rank_rev'] = df.groupby('user_idx').cumcount(ascending=False)
user_counts    = df.groupby('user_idx')['item_idx'].transform('count')

df['split'] = 'train'
df.loc[(df['rank_rev'] == 0) & (user_counts >= 3), 'split'] = 'test'
df.loc[(df['rank_rev'] == 1) & (user_counts >= 3), 'split'] = 'val'
df = df.drop(columns=['rank_rev'])

df[['user_idx', 'item_idx', 'timestamp', 'split']].to_csv(
    os.path.join(BEAUTY_PATH, 'interactions.csv'), index=False)
items[['parent_asin', 'item_idx']].to_csv(
    os.path.join(BEAUTY_PATH, 'item_map.csv'), index=False)
pd.DataFrame({'user_id': uid, 'user_idx': range(len(uid))}).to_csv(
    os.path.join(BEAUTY_PATH, 'user_map.csv'), index=False)

ITEM_COUNT = int(items['item_idx'].max())
print('✔  Process complete')
print(df['split'].value_counts().to_string())
print(f'Users : {df["user_idx"].nunique():,}')
print(f'Items : {df["item_idx"].nunique():,}  (ITEM_COUNT={ITEM_COUNT})')

test_users = df[df['split'] == 'test']['user_idx'].nunique()
print(f'Test users (should be > 0): {test_users:,}')


✔  Process complete
split
train    677545
val        8151
test       8151
Users : 631,915
Items : 112,553  (ITEM_COUNT=112553)
Test users (should be > 0): 8,151


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class MultimodalFusion(nn.Module):
    """Gated cross-attention fusion of text, image, and ID embeddings."""

    def __init__(self, text_dim=384, image_dim=2048, inter_dim=128, shared_dim=128):
        super().__init__()
        self.text_proj  = nn.Linear(text_dim,  shared_dim)
        self.image_proj = nn.Linear(image_dim, shared_dim)
        self.inter_proj = nn.Linear(inter_dim, shared_dim)

        self.attn_i2t = nn.MultiheadAttention(shared_dim, num_heads=4, batch_first=True)
        self.attn_t2i = nn.MultiheadAttention(shared_dim, num_heads=4, batch_first=True)

        self.gate = nn.Sequential(nn.Linear(shared_dim * 2, shared_dim), nn.Sigmoid())
        self.out  = nn.Linear(shared_dim, shared_dim)

    def forward(self, text_emb, image_emb, interaction_emb):
        t = self.text_proj(text_emb).unsqueeze(1)
        v = self.image_proj(image_emb).unsqueeze(1)
        r = self.inter_proj(interaction_emb)

        v2t = self.attn_i2t(v, t, t)[0].squeeze(1)
        t2v = self.attn_t2i(t, v, v)[0].squeeze(1)

        g = self.gate(torch.cat([v2t, t2v], dim=-1))
        return self.out(g * v2t + (1 - g) * t2v + r)

In [ ]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

class TiSASRecDataset(Dataset):
    def __init__(
        self,
        df,
        mode:    str = 'train',
        max_len: int = 50,
        num_neg: int = 99,
        seed:    int = 42,
    ):
        assert mode in ('train', 'val', 'test')
        self.max_len = max_len
        self.mode    = mode
        self.num_neg = num_neg
        self.rng     = np.random.default_rng(seed)

        train_df = df[df['split'] == 'train']

        train_seqs  = train_df.groupby('user_idx')['item_idx'].apply(list)
        train_times = train_df.groupby('user_idx')['timestamp'].apply(list)
        all_items = np.arange(1, df['item_idx'].max() + 1)
        user_item_sets = (
            df.groupby('user_idx')['item_idx']
            .apply(set).to_dict()
        )

        self.all_items = all_items
        self.records   = []

        if mode == 'train':
            for uid, seq in train_seqs.items():
                if len(seq) < 2:
                    continue
                self.records.append({
                    'history':  seq[:-1],
                    'times':    train_times[uid][:-1],
                    'target':   seq[-1],
                    'seen':     user_item_sets[uid],
                })
        else:
            target_df   = df[df['split'] == mode]
            tgt_items   = target_df.groupby('user_idx')['item_idx'].first().to_dict()
            tgt_times   = target_df.groupby('user_idx')['timestamp'].first().to_dict()

            for uid, target in tgt_items.items():
                seq = train_seqs.get(uid)
                if seq is None or len(seq) < 1:
                    continue
                self.records.append({
                    'history': seq,
                    'times':   train_times[uid],
                    'target':  target,
                    'seen':    user_item_sets[uid],
                })

        print(f'[TiSASRecDataset:{mode}] {len(self.records):,} users')

    def _sample_negatives(self, seen, n):
        negs = []
        while len(negs) < n:
            cands = self.rng.choice(self.all_items, size=n * 2, replace=False)
            negs  += [c for c in cands.tolist() if c not in seen]
        return negs[:n]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec  = self.records[idx]
        hist = rec['history'][-(self.max_len):]
        ts   = rec['times']  [-(self.max_len):]
        raw_iv    = [ts[i+1] - ts[i] for i in range(len(ts) - 1)] + [0.0]
        intervals = [float(np.log1p(max(dt, 0))) for dt in raw_iv]

        L     = len(hist)
        s_arr = np.zeros(self.max_len, dtype=np.int64)
        i_arr = np.zeros(self.max_len, dtype=np.float32)
        s_arr[-L:] = hist
        i_arr[-L:] = intervals
        target = rec['target']
        if self.mode == 'train':
            neg = self._sample_negatives(rec['seen'], 1)[0]
            return (
                torch.from_numpy(s_arr),
                torch.from_numpy(i_arr),
                torch.tensor(target, dtype=torch.long),
                torch.tensor(neg,    dtype=torch.long),
            )
        else:
            negs = self._sample_negatives(rec['seen'], self.num_neg)
            candidates = torch.tensor([target] + negs, dtype=torch.long)
            return (
                torch.from_numpy(s_arr),
                torch.from_numpy(i_arr),
                candidates,
            )

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

class MultiModalTiSASRec(nn.Module):
      def __init__(self, item_count, shared_dim, img_path, txt_path, max_len=50):
        super().__init__()
        self.max_len = max_len

        img_np = np.load(img_path).astype(np.float32)
        txt_np = np.load(txt_path).astype(np.float32)
        img_np /= np.linalg.norm(img_np, axis=1, keepdims=True).clip(min=1e-8)
        txt_np /= np.linalg.norm(txt_np, axis=1, keepdims=True).clip(min=1e-8)
        self.register_buffer('img_feats', torch.from_numpy(img_np))
        self.register_buffer('txt_feats', torch.from_numpy(txt_np))

        self.fusion   = MultimodalFusion(shared_dim=shared_dim)
        self.item_emb = nn.Embedding(item_count + 1, shared_dim, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, shared_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, shared_dim), nn.ReLU(), nn.Linear(shared_dim, shared_dim)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=shared_dim, nhead=8, batch_first=True,
            dim_feedforward=shared_dim * 4, dropout=0.2, norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.norm        = nn.LayerNorm(shared_dim)

    def encode_sequence(self, seq, intervals):
        B, S = seq.shape
        img  = F.pad(self.img_feats, (0, 0, 1, 0))[seq]
        txt  = F.pad(self.txt_feats, (0, 0, 1, 0))[seq]
        ids  = self.item_emb(seq)

        fused = self.fusion(
            txt.reshape(-1, txt.shape[-1]),
            img.reshape(-1, img.shape[-1]),
            ids.reshape(-1, ids.shape[-1]),
        ).reshape(B, S, -1)

        time_enc = self.time_mlp(intervals.unsqueeze(-1))
        pos_enc  = self.pos_emb(torch.arange(S, device=seq.device))
        x        = self.norm(fused + time_enc + pos_enc)

        causal_mask = torch.triu(torch.ones(S, S, device=seq.device), 1).bool()
        out         = self.transformer(x, mask=causal_mask)
        return out[:, -1, :]

    def score_candidates(self, user_repr, candidates):
          cand_emb = self.item_emb(candidates)
        return torch.bmm(cand_emb, user_repr.unsqueeze(-1)).squeeze(-1)

    def forward(self, seq, intervals, candidates):
        user_repr = self.encode_sequence(seq, intervals)
        return self.score_candidates(user_repr, candidates)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader

MAX_LEN    = 50
NUM_EPOCHS = 30
BATCH_SIZE = 256

train_dataset = TiSASRecDataset(df, mode='train', max_len=MAX_LEN)
train_loader  = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
)

model = MultiModalTiSASRec(
    item_count = ITEM_COUNT,
    shared_dim = 128,
    max_len    = MAX_LEN,
    img_path   = os.path.join(BEAUTY_PATH, 'image_embeddings.npy'),
    txt_path   = os.path.join(BEAUTY_PATH, 'text_embeddings.npy'),
).cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-5
)

def bpr_loss(pos_score, neg_score):
    """Bayesian Personalised Ranking loss. Maximises pos - neg margin."""
    return -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, n_batches = 0.0, 0

    for seq, inter, pos, neg in train_loader:
        seq, inter = seq.cuda(), inter.cuda()
        pos, neg   = pos.cuda(), neg.cuda()

        optimizer.zero_grad()
        candidates = torch.stack([pos, neg], dim=1)
        scores     = model(seq, inter, candidates)
        loss = bpr_loss(scores[:, 0], scores[:, 1])
        if torch.isnan(loss):
            print(f'  ⚠  NaN at epoch {epoch} – skipping batch')
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1

    scheduler.step()
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | BPR Loss: {total_loss/max(n_batches,1):.4f}'
          f' | LR: {scheduler.get_last_lr()[0]:.2e}')

[TiSASRecDataset:train] 37,571 users
Epoch 01/30 | BPR Loss: 4.8940 | LR: 9.97e-04
Epoch 02/30 | BPR Loss: 3.6986 | LR: 9.89e-04
Epoch 03/30 | BPR Loss: 2.6529 | LR: 9.76e-04
Epoch 04/30 | BPR Loss: 1.7276 | LR: 9.57e-04
Epoch 05/30 | BPR Loss: 0.9374 | LR: 9.34e-04
Epoch 06/30 | BPR Loss: 0.6592 | LR: 9.05e-04
Epoch 07/30 | BPR Loss: 0.5826 | LR: 8.73e-04
Epoch 08/30 | BPR Loss: 0.5157 | LR: 8.36e-04
Epoch 09/30 | BPR Loss: 0.4649 | LR: 7.96e-04
Epoch 10/30 | BPR Loss: 0.4169 | LR: 7.53e-04
Epoch 11/30 | BPR Loss: 0.3681 | LR: 7.06e-04
Epoch 12/30 | BPR Loss: 0.3247 | LR: 6.58e-04
Epoch 13/30 | BPR Loss: 0.2844 | LR: 6.08e-04
Epoch 14/30 | BPR Loss: 0.2507 | LR: 5.57e-04
Epoch 15/30 | BPR Loss: 0.2190 | LR: 5.05e-04
Epoch 16/30 | BPR Loss: 0.2000 | LR: 4.53e-04
Epoch 17/30 | BPR Loss: 0.1698 | LR: 4.02e-04
Epoch 18/30 | BPR Loss: 0.1505 | LR: 3.52e-04
Epoch 19/30 | BPR Loss: 0.1367 | LR: 3.04e-04
Epoch 20/30 | BPR Loss: 0.1223 | LR: 2.58e-04
Epoch 21/30 | BPR Loss: 0.1150 | LR: 2.14e-

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
train_item_counts = (
    df[df['split'] == 'train']['item_idx'].value_counts().to_dict()
)

NUM_NEG   = 99
test_dataset = TiSASRecDataset(df, mode='test', max_len=MAX_LEN, num_neg=NUM_NEG)
test_loader  = DataLoader(
    test_dataset, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True,
)

def calculate_metrics(model, loader, train_counts, ks=(5, 10, 20)):
    model.eval()
    tiers = ('cold', 'few_shot', 'warm')
    results = {t: {f'recall@{k}': [] for k in ks} for t in tiers}
    for t in tiers:
        for k in ks:
            results[t][f'ndcg@{k}'] = []

    with torch.no_grad():
        for seq, inter, candidates in loader:
            seq, inter       = seq.cuda(), inter.cuda()
            candidates_cuda  = candidates.cuda()
            scores    = model(seq, inter, candidates_cuda)
            scores_np = scores.cpu().numpy()

            for i in range(len(candidates)):
                target_item = candidates[i, 0].item()
                score_pos   = scores_np[i, 0]
                scores_neg  = scores_np[i, 1:]
                rank = int((scores_neg >= score_pos).sum()) + 1
                count = train_counts.get(target_item, 0)
                tier  = 'cold' if count == 0 else ('few_shot' if count < 5 else 'warm')
                for k in ks:
                    results[tier][f'recall@{k}'].append(1 if rank <= k else 0)
                    results[tier][f'ndcg@{k}'].append(
                        (1.0 / np.log2(rank + 1)) if rank <= k else 0.0
                    )
    k_headers = ''.join(f" | {'R@'+str(k):<8} {'N@'+str(k):<8}" for k in ks)
    header    = f"{'Tier':<12}{k_headers} | Samples"
    sep       = '─' * len(header)
    print('\n' + sep)
    print(header)
    print(sep)
    for tier in tiers:
        row = f'{tier:<12}'
        n   = len(results[tier][f'recall@{ks[0]}'])
        for k in ks:
            r = np.mean(results[tier][f'recall@{k}']) if n else 0.0
            d = np.mean(results[tier][f'ndcg@{k}'])   if n else 0.0
            row += f' | {r:<8.4f} {d:<8.4f}'
        row += f' | {n}'
        print(row)
    print(sep)

    all_r = {k: [] for k in ks}
    all_n = {k: [] for k in ks}
    for tier in tiers:
        for k in ks:
            all_r[k] += results[tier][f'recall@{k}']
            all_n[k] += results[tier][f'ndcg@{k}']
    row = f"{'overall':<12}"
    for k in ks:
        r = np.mean(all_r[k]) if all_r[k] else 0.0
        d = np.mean(all_n[k]) if all_n[k] else 0.0
        row += f' | {r:<8.4f} {d:<8.4f}'
    row += f' | {len(all_r[ks[0]])}'
    print(row)
    print(sep)

print('Evaluating on test set (100-way sampled ranking)...')
calculate_metrics(model, test_loader, train_item_counts)

[TiSASRecDataset:test] 8,151 users
Evaluating on test set (100-way sampled ranking)...

──────────────────────────────────────────────────────────────────────────────────
Tier         | R@5      N@5      | R@10     N@10     | R@20     N@20     | Samples
──────────────────────────────────────────────────────────────────────────────────
cold         | 0.0010   0.0005   | 0.0153   0.0049   | 0.0842   0.0218   | 1045
few_shot     | 0.0378   0.0208   | 0.0922   0.0380   | 0.2021   0.0654   | 1930
warm         | 0.3331   0.2328   | 0.4635   0.2747   | 0.5875   0.3063   | 5176
──────────────────────────────────────────────────────────────────────────────────
overall      | 0.2206   0.1528   | 0.3181   0.1841   | 0.4317   0.2128   | 8151
──────────────────────────────────────────────────────────────────────────────────
